## A notebook to analyze groups of croparrays  

### Read in, combine, and making measurements 

In [1]:
import croparray as ca
from pathlib import Path
%gui qt

In [ ]:
# ============================
# USER-DEFINED PARAMETERS
# ============================

# Files & Directories
DATA_DIR_1 = Path("C:/Users/tstasevi/Documents/Tim-Marianas/TrnlWithZNF598/20260210/NoZNF598/Stacks/CropArrays")
DATA_DIR_2 = Path("C:/Users/tstasevi/Documents/Tim-Marianas/TrnlWithZNF598/20260210/WithZNF598/HT/Stacks/CropArrays")
LABELS = ['-ZNF598','+ZNF598']
OUTPUT = Path("C:/Users/tstasevi/Documents/Tim-Marianas/TrnlWithZNF598/20260210/HT-analysis") 

nc_files_1 = sorted(DATA_DIR_1.glob("*.nc"))
nc_files_2 = sorted(DATA_DIR_2.glob("*.nc"))

# Measurement parameters
REF_CH = 0  # Optional reference channel for best_z_proj (if use_zc=False, i.e. when Z not tracked)
DISK_R = 5  # Radius of disk in pixels to measure signal
DISK_BG = 7 # Radius of single pixel ring around disk to measure background 
ROLL_N = 1  # Rolling z-average for signal measurement.

In [3]:
nc_files_2

[WindowsPath('C:/Users/tstasevi/Documents/Tim-Marianas/TrnlWithZNF598/20260210/WithZNF598/HT/Stacks/CropArrays/Max_ALL_Cell_HT - Position 1_XY1770783062.ome.nc'),
 WindowsPath('C:/Users/tstasevi/Documents/Tim-Marianas/TrnlWithZNF598/20260210/WithZNF598/HT/Stacks/CropArrays/Max_ALL_Cell_HT - Position 2_XY1770783063.ome.nc'),
 WindowsPath('C:/Users/tstasevi/Documents/Tim-Marianas/TrnlWithZNF598/20260210/WithZNF598/HT/Stacks/CropArrays/Max_ALL_Cell_HT - Position 3_XY1770783064.ome.nc'),
 WindowsPath('C:/Users/tstasevi/Documents/Tim-Marianas/TrnlWithZNF598/20260210/WithZNF598/HT/Stacks/CropArrays/Max_ALL_Cell_HT - Position 4_XY1770783065.ome.nc'),
 WindowsPath('C:/Users/tstasevi/Documents/Tim-Marianas/TrnlWithZNF598/20260210/WithZNF598/HT/Stacks/CropArrays/Max_ALL_Cell_HT - Position 5_XY1770783066.ome.nc'),
 WindowsPath('C:/Users/tstasevi/Documents/Tim-Marianas/TrnlWithZNF598/20260210/WithZNF598/HT/Stacks/CropArrays/Max_ALL_Cell_HT - Position 6_XY1770783067.ome.nc'),
 WindowsPath('C:/Users

In [11]:
my_ca = ca.build.open_measure_concat(
    groups=[nc_files_1, nc_files_2],
    dims=["exp", "fov"],
    labels=[["-ZNF598", "+ZNF598"], None],
    measure_kwargs=dict(ref_ch=REF_CH, disk_r=DISK_R, disk_bg=DISK_BG, roll_n=ROLL_N,drop_int=True, # drop full z-stack to save memory, keeping best_z_proj
    ),
    open_as = "croparray",  # can be croparray or trackarray 
    join="outer",
)

In [12]:
my_ca.ds

<xarray.Dataset> Size: 1GB
Dimensions:          (exp: 2, fov: 14, n: 278, t: 46, ch: 1, y: 21, x: 21, z: 3)
Coordinates:
  * n                (n) int16 556B 0 1 2 3 4 5 6 ... 272 273 274 275 276 277
  * t                (t) int32 184B 0 1 2 3 4 5 6 7 ... 38 39 40 41 42 43 44 45
  * y                (y) float64 168B -0.735 -0.6615 -0.588 ... 0.6615 0.735
  * x                (x) float64 168B -0.735 -0.6615 -0.588 ... 0.6615 0.735
  * z                (z) float64 24B -0.35 0.0 0.35
  * ch               (ch) int32 4B 0
  * fov              (fov) <U45 3kB 'Max_ALL_Cell_HT - Position 1_XY177078306...
  * exp              (exp) <U7 56B '-ZNF598' '+ZNF598'
Data variables: (12/20)
    xc               (exp, fov, n, t, ch) float32 1MB nan nan nan ... nan nan
    yc               (exp, fov, n, t, ch) float32 1MB nan nan nan ... nan nan
    zc               (exp, fov, n, t, ch) float32 1MB nan nan nan ... nan nan
    xc_pix           (exp, fov, n, t, ch) float32 1MB nan nan nan ... nan nan
    yc_pix           (exp, fov, n, t, ch) float32 1MB nan nan nan ... nan nan
    zc_pix           (exp, fov, n, t, ch) float32 1MB nan nan nan ... nan nan
    ...               ...
    id               (exp, fov, n, t) float32 1MB nan nan nan ... nan nan nan
    z_pos_best       (exp, fov, n, t) float32 1MB nan nan nan ... nan nan nan
    zc_best_pix      (exp, fov, n, t, ch) float32 1MB nan nan nan ... nan nan
    zc_best          (exp, fov, n, t, ch) float32 1MB nan nan nan ... nan nan
    best_z           (exp, ch, fov, n, t, y, x) float64 1GB nan nan ... nan nan
    signal           (exp, ch, fov, n, t) float64 3MB nan nan nan ... nan nan
Attributes: (12/15)
    name:                      Max_ALL_HI_at_5 - Position 1_XY1770772846.ome ...
    date:                      2026-02-10
    xy_pad:                    10
    z_pad:                     1
    dx:                        0.0735
    dy:                        0.0735
    ...                        ...
    t_units:                   min
    signal_units:              a.u.
    croparray_schema_version:  2.0
    notes:                     A crop array with ch0 = 12xSunTag-KDM5B-XBP1(S...
    provenance_json:           {\n  "timestamp": "2026-03-03T17:26:44",\n  "d...
    concat_meta_json:          {"base_name": "Max_ALL_HI_at_5_-_Position_1_XY...

In [13]:
# Create binary masks from "best_z" so can measure morphological properties. 
my_ca.ops.apply(
    ca.tools.binarize_crop_manual,
    channels=[0],               # Channel used to generate mask
    source="best_z",            # Image layer to threshold (typically z-projected signal)
    out_name="ch{ch}_mask",     # Output mask variable name (→ "ch0_mask")

    # Recommended parameters forwarded directly to binarize_crop_manual(...)
    func_kwargs=dict(
        q=0.45,                 # Threshold at 45% after intensity normalization
        q_range=(0.02, 0.999),  # Normalize intensities using 2%–99.9% quantile range
        q_positive_only=True,   # Ignore negative pixels (important for best_z)
        close_px=1,             # Morphological closing (bridge small gaps)
        smooth_px=0,            # Light smoothing of mask edges
        fill_holes=False,       # Do not fill internal holes
        return_uint8=True,      # Store mask as uint8 (0/1)
        morph_px=0,             # Apply a dilation of final mask (morph_pix > 0; <0 for erosion)
    ),
);

In [14]:
# Check masks with napari
viewer, layers = my_ca.napari.montage_viewer(
    row="n",
    col="t",
    show=("best_z", "ch0_mask"),
    #ch=[0, 1, 2],  # RGB
    colormaps={"ch0_mask": "magenta","best_z": "green"},
    image_contrast=[0.5,99.5],
    show_tile_text=False,
)